# Submissão 3 - Modelo End-to-End (5 Classes Simultâneas)
**Problemas resolvidos neste notebook:**
1. Fim dos Embeddings Congelados: O DeBERTa vai treinar juntamente com a DNN (*Fine-Tuning* completo).
2. Fim do Gargalo Hierárquico: Prevemos as 5 classes diretamente (Humano, Google, Meta, Mistral, OpenAI), acabando com os erros em cascata do modelo Binário.
3. Optimização para VRAM de 2GB (Placa MX330): Uso de *Gradient Accumulation* e Mixed Precision (FP16).

In [ ]:
!pip install transformers accelerate evaluate -q
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, accuracy_score
from torch.cuda.amp import autocast, GradScaler
import os

DATA_DIR = "../data"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

df_train = pd.read_csv(os.path.join(DATA_DIR, "dataset_treino.csv"))
df_val = pd.read_csv(os.path.join(DATA_DIR, "dataset_validacao.csv"))

# Ponto 2 e 4: Sem clean_text, Sem Hierarquia (Flat 5 Labels)
df_train['Text'] = df_train['Text'].astype(str)
df_val['Text'] = df_val['Text'].astype(str)

unique_labels = sorted(df_train['source_name'].dropna().unique().tolist())
label_mapping = {k: v for v, k in enumerate(unique_labels)}
reverse_mapping = {v: k for k, v in label_mapping.items()}
print("Mapeamento Global (5 Classes):", label_mapping)

df_train['label'] = df_train['source_name'].map(label_mapping)
df_val['label'] = df_val['source_name'].map(label_mapping)

In [ ]:
# Ponto 3: Uso de DeBERTa v3 small (Case-Sensitive e Melhor que o DistilBERT)
MODEL_NAME = 'microsoft/deberta-v3-small'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts.tolist(), 
            truncation=True, 
            padding='max_length', 
            max_length=max_length
        )
        self.labels = labels.tolist()
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

# BATCH LIMITADA PARA A PLACA MX330 2GB
# Vamos usar o Gradient Accumulation mais abaixo para simular 32 de batch size
BATCH_SIZE = 4 

train_dataset = TextDataset(df_train['Text'], df_train['label'], tokenizer)
val_dataset = TextDataset(df_val['Text'], df_val['label'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2)

In [ ]:
# Ponto 1: Arquitetura End-to-End (O Transformer liga diretamente à DNN e treina em conjunto)
class EndToEndDetector(nn.Module):
    def __init__(self, model_name, num_classes, hidden_size=128):
        super(EndToEndDetector, self).__init__()
        # Extrator (Transformer)
        self.transformer = AutoModel.from_pretrained(model_name)
        
        # Opcional para gráficas muito fracas: Congelar os embeddings originais para poupar VRAM
        for param in self.transformer.embeddings.parameters():
            param.requires_grad = False
        # Para poupar VRAM na vossa MX330, garantimos que treinamos apenas as ultimas layers do Transformer!
        for layer in self.transformer.encoder.layer[:-2]:
            for param in layer.parameters():
                param.requires_grad = False
                
        # DNN Classifier da Submissão 2 conectada
        self.fc1 = nn.Linear(self.transformer.config.hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        # Passar raw text pelo DeBERTa
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        # Extrair representação global (Mean Pooling do DeBERTa ou CLS token dependendo da layer final)
        # DeBERTa não tem CLS isolado pronto como o BERT, então usamos o "mean token" sobre toda a sequence
        mask_expanded = attention_mask.unsqueeze(-1).expand(out.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(out.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        # Passar pela DNN (Classificação 5 classes conjunta)
        x = self.dropout(self.relu(self.fc1(mean_pooled)))
        logits = self.fc2(x)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo a ser usado:", device)

num_classes = len(unique_labels)
model = EndToEndDetector(MODEL_NAME, num_classes=num_classes).to(device)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR

# Parâmetros de Treino
EPOCHS = 3
LEARNING_RATE = 2e-5
# Mixed Precision Trick (Simula Batch Size de 32 mesmo com 4)
ACCUMULATION_STEPS = 8 

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler() # Para poupar VRAM e acelerar na NVIDIA (Mixed Precision FP16)

print("Começar Treino Otimizado End-to-End para a GPU local...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for i, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Ativar Mixed Precision (FP16)
        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            # Dividir a loss pelo accumulation_steps (preparação para a mega soma)
            loss = loss / ACCUMULATION_STEPS
            
        scaler.scale(loss).backward() # Backpropagation Flui até às layers decongeladas do Transformer!
        
        # Atualizar Pesos só quando atingimos os N steps acumulados (Simula super Batch)
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        total_loss += loss.item() * ACCUMULATION_STEPS
        
        if (i+1) % 400 == 0:
            print(f"Época {epoch+1} | Batch {i+1}/{len(train_loader)} | Loss: {total_loss/(i+1):.4f}")
            
    print(f"Época {epoch+1} Concluída - Média Estimada de Loss: {total_loss/len(train_loader):.4f}")

# Cuidado, demora cerca de 1 hora a treinar numa MX330.
print("A salvar modelo Submissão 3...")
torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'subm3_deberta_endtoend.pt'))

In [ ]:
# Validação Simples
print("A calcular precisão Final!")
model.eval()
predictions = []
real_labels = []

# No testing the gradients aren't saved, so GPU will manage well.
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        with autocast():
            logits = model(input_ids, attention_mask)
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(preds)
        real_labels.extend(batch['labels'].numpy())

print("\n=== Relatório de Avaliação (Flat 5 Classes) ===")
print(classification_report(real_labels, predictions, target_names=unique_labels))
print("Accuracy Global:", accuracy_score(real_labels, predictions))

# Exportar Submissão
df_val['predicted_source'] = [reverse_mapping[p] for p in predictions]
os.makedirs('../subm3', exist_ok=True)
df_val.to_csv('../subm3/subm3_deberta_preds.csv', index=False)
